# Harvest DataCite dataset records

## Import

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
from sindex.sources.datacite.jobs import (
    harvest_datacite_datasets_for_date_range_to_ndjson,
    batch_slim_datacite_record_to_ndjson,
    batch_slim_datacite_record_to_ndjson_fast,
    batch_slim_datacite_chunked #FASTEST
    )
from sindex.utils.datasets import create_datasets_db_from_ndjson
from pathlib import Path
import os
import duckdb

## Query datacite and write to ndjson files

In [9]:
start_date_str = "2025-10-01"
end_date_str = "2025-10-31" #update this date if code fails to the one that was running
out_dir = Path(r"D:\may-2026-data\records\raw-records\datacite-records")

out_dir.mkdir(parents=True, exist_ok=True)

n = harvest_datacite_datasets_for_date_range_to_ndjson(
    start_date_str=start_date_str,
    end_date_str=end_date_str,
    window_days=7,
    page_size=1000,
    save_folder=str(out_dir),
    skip_empty_files=True,
    polite_sleep_seconds=0.5,
)

print(f"Done. Wrote {n:,} records across window files in {out_dir.resolve()}")

Fetching records 2025-10-25 → 2025-10-31 (window_days=7, page_size=1000, detail=True)
  Saved 775200 records → D:\may-2026-data\records\raw-records\datacite-records\datacite-2025-10-25-2025-10-31.ndjson
Fetching records 2025-10-18 → 2025-10-24 (window_days=7, page_size=1000, detail=True)
  Saved 325467 records → D:\may-2026-data\records\raw-records\datacite-records\datacite-2025-10-18-2025-10-24.ndjson
Fetching records 2025-10-11 → 2025-10-17 (window_days=7, page_size=1000, detail=True)
  Saved 2504949 records → D:\may-2026-data\records\raw-records\datacite-records\datacite-2025-10-11-2025-10-17.ndjson
Fetching records 2025-10-04 → 2025-10-10 (window_days=7, page_size=1000, detail=True)
  Saved 1711799 records → D:\may-2026-data\records\raw-records\datacite-records\datacite-2025-10-04-2025-10-10.ndjson
Fetching records 2025-10-01 → 2025-10-03 (window_days=7, page_size=1000, detail=True)
  Saved 20687 records → D:\may-2026-data\records\raw-records\datacite-records\datacite-2025-10-01-20

---------------
---------------
Check results on DataCite commons: https://commons.datacite.org/doi.org?query=%28types.resourceTypeGeneral%3ADataset%29+AND+%28created%3A%5B2025-09-24+TO+2025-09-30%5D%29&registration-agency=datacite

## Write batch slim records

In [10]:
src_folder = r"D:\may-2026-data\records\raw-records\datacite-records"
dst_folder =r"D:\may-2026-data\records\slim-records\datacite-slim-records"

In [11]:
summary = batch_slim_datacite_chunked(
    src_folder=src_folder,
    dst_folder=dst_folder)

Found 32 input files.
Starting processing with 32 cores. Batch size: 100,000...
Processed: 21,274,127 lines | Batches: 213 | Bad: 0
 
DONE in 9212.32s
Total Lines Read: 21,274,127
Total Lines Kept: 21,274,127
Processing Rate:  2,309 records/sec
Output Files:     213 files written to D:\may-2026-data\records\slim-records\datacite-slim-records
 


## Save new dataset info to duckdb DB

This is needed to find topics and citations from OpenAlex more easily with DuckDB.
Run every time new datasets are pulled from DataCite to add to the existing table.

In [1]:
slim_folder = r"D:\may-2026-data\records\slim-records\datacite-slim-records"
db_path = r"D:\combined-data\records\slim-records\datacite-slim-records.duckdb"

In [6]:
create_datasets_db_from_ndjson(slim_folder, db_path)

Extracting data from 213 files


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Success! DB at: D:\combined-data\records\slim-records\datacite-slim-records.duckdb
New datasets added: 0
Total datasets in DB: 70,283,649


In [9]:
con = duckdb.connect(db_path)
row_count = con.execute("SELECT count() FROM my_datasets").fetchone()[0]
print(f"Total datasets in table: {row_count:,}")
display(con.execute("SELECT * FROM my_datasets LIMIT 5").df())
con.close()

Total datasets in table: 70,283,649


,dataset_id,id_type,pubyear,added_date
0,10.5284/1000389,doi,2011,2026-01-01
1,10.5284/1000140,doi,2011,2026-01-01
2,10.5284/1000146,doi,2011,2026-01-01
3,10.5284/1000144,doi,2011,2026-01-01
4,10.5284/1000181,doi,2011,2026-01-01


### Export dataset_ids as csv for Software Heritage processing

In [5]:
output_path = r"D:\may-2026-data\records\ids\datacite_ids.csv"

with duckdb.connect(db_path) as con:
    con.execute(f"""
        COPY (
            SELECT dataset_id 
            FROM my_datasets 
            WHERE added_date >= '2026-05-01'
        ) TO '{output_path.replace(chr(92), "/")}' (FORMAT CSV, HEADER true)
    """)
    print("Export complete.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Export complete.


## ------ EXTRA ------

## Number of data repositories/publishers

In [5]:
input_pattern = r'D:\pipeline-data\records\slim-records\datacite-slim-records\*.ndjson'
output_file = r'D:\pipeline-data\records\publisher_counts.csv'

In [ ]:
import json
import glob
import csv
import sys
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

def process_single_file(file_path):
    local_counts = Counter()
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line: 
                    continue
                try:
                    data = json.loads(line)
                    raw_publisher = data.get('publisher')
                    
                    if raw_publisher:
                        pub = str(data.get('publisher') or 'unknown').strip().lower()
                        local_counts[pub] += 1
                    else:
                        local_counts['Unknown'] += 1
                        
                except json.JSONDecodeError:
                    continue
    except Exception as e:
        print(f"\nSkipping file {file_path}: {e}")
    return local_counts


files = glob.glob(input_pattern)
global_counts = Counter()
processed_count = 0

print(f"Processing {len(files)} files...")

with ThreadPoolExecutor(max_workers=8) as executor:
    future_to_file = {executor.submit(process_single_file, f): f for f in files}
    
    for future in as_completed(future_to_file):
        result_counter = future.result()
        global_counts.update(result_counter)
        processed_count += 1
        
        if processed_count % 10 == 0:
            sys.stdout.write(f"\rFiles: {processed_count}/{len(files)}")
            sys.stdout.flush()

with open(output_file, 'w', newline='', encoding='utf-8-sig') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['Publisher', 'Dataset Count'])
    for pub, count in global_counts.most_common():
        writer.writerow([pub, count])

print(f"\nDone! Saved to {output_file} .")
print(f"\n\nSuccess! Final unique count: {len(global_counts)}")

Processing 771 files...
Files: 420/771

## Number of rights (License)

In [3]:
input_pattern = r'D:\pipeline-data\records\raw-records\datacite-records\*.ndjson'
output_file = r'D:\pipeline-data\records\license_counts.csv'

In [5]:
import json
import csv
import glob
import sys
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

def process_single_file(file_path):
    local_counts = Counter()
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line: 
                    continue
                try:
                    data = json.loads(line)
                    attributes = data.get('attributes', {})
                    rights_list = attributes.get('rightsList', [])

                    if rights_list:
                        for item in rights_list:
                            right_text = item.get('rights')
                            if right_text:
                                clean_right = str(right_text).strip()
                                local_counts[clean_right] += 1
                    else:
                        local_counts['No Rights Listed'] += 1
                        
                except json.JSONDecodeError:
                    continue
    except Exception as e:
        print(f"\nSkipping file {file_path}: {e}")
    return local_counts

files = glob.glob(input_pattern)
global_counts = Counter()
processed_count = 0

print(f"Processing {len(files)} files")

with ThreadPoolExecutor(max_workers=8) as executor:
    future_to_file = {executor.submit(process_single_file, f): f for f in files}
    
    for future in as_completed(future_to_file):
        result_counter = future.result()
        global_counts.update(result_counter)
        processed_count += 1
        
        if processed_count % 10 == 0:
            sys.stdout.write(f"\rFiles: {processed_count}/{len(files)}")
            sys.stdout.flush()

# Write to CSV
with open(output_file, 'w', newline='', encoding='utf-8-sig') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['License', 'Count'])
    for right, count in global_counts.most_common():
        writer.writerow([right, count])

print(f"\nDone! Saved to {output_file}.")
print(f"Final unique rights count: {len(global_counts)}")

Processing 771 files
Files: 770/771
Done! Saved to D:\pipeline-data\records\license_counts.csv.
Final unique rights count: 241445
